In [ ]:
!pip install deepface

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
from deepface import DeepFace
from deepface.modules.exceptions import FaceNotDetected

try:
  # Take a photo.
  image_file = take_photo()

  # Detect faces and estimate emotions.
  result = DeepFace.analyze(img_path = image_file, actions = ['emotion'])

  # Print the number of persons detected.
  print('%d person(s) detected.' % len(result))

  # Print the probability of each emotion for each person.
  for i in range(len(result)):
    print('Person #%d' % (i + 1))
    for e, p in result[i]['emotion'].items():
      print('%s: %f' % (e, p))

except FaceNotDetected:
  # If no face is detected, a FaceNotDetected exception is raised.
  print('No face detected.')

except Exception as e:
  # If any other exceptions are raised, details will be displayed.
  print(e)